# Validate Docker Compose (Spark + Jupyter)

Run this **inside** the `jupyter` service (`make notebook` → Lab at http://localhost:8888). It checks that PySpark reaches the standalone master and runs a small job.

**Using `latest` images:** run `docker compose pull` for **all** services together, then `docker compose up -d`, so the PySpark in `jupyter/pyspark-notebook:latest` matches the Spark version on `apache/spark:latest`. Mixed versions often show up as connection errors to `spark-master:7077`.

**After `spark.stop()`:** use **Kernel → Restart Kernel**, then run all cells from the top. A second `SparkContext` in the same process often hits driver issues (for example `MetricsSystem` / `getServletHandlers`).

In [1]:
import os

# Do not set PYSPARK_SUBMIT_ARGS here — overriding image defaults can break Py4J (JAVA_GATEWAY_EXITED).

import pyspark
from pyspark.sql import SparkSession

master = os.environ.get("SPARK_MASTER", "spark://spark-master:7077")
print("SPARK_MASTER:", master)
print("PySpark (Python package):", getattr(pyspark, "__version__", "unknown"))

_STOPPED_FLAG = "PYSPARK_EXAMPLE_SESSION_STOPPED"


def spark_session() -> SparkSession:
    """Return a SparkSession for this kernel. Do not use after `spark.stop()` in the same kernel — restart the kernel instead."""
    if os.environ.get(_STOPPED_FLAG) == "1":
        raise RuntimeError(
            "Spark was stopped in this kernel. Use Kernel → Restart Kernel, then run all cells from the top. "
            "A second SparkContext in the same Jupyter process often fails with MetricsSystem / getServletHandlers."
        )
    s = SparkSession.getActiveSession()
    if s is not None and s._jsc is not None and not s._jsc.sc().isStopped():
        return s
    if SparkSession._instantiatedSession is not None:
        raise RuntimeError(
            "Stale SparkSession in this kernel. Kernel → Restart Kernel, then run all cells from the top."
        )
    return (
        SparkSession.builder.appName("docker-compose-validation")
        .master(master)
        .config("spark.ui.enabled", "false")
        .config("spark.ui.showConsoleProgress", "false")
        .getOrCreate()
    )


SPARK_MASTER: spark://spark-master:7077


In [2]:
spark = spark_session()

print("Spark version (JVM driver):", spark.version)
print("Master (runtime):", spark.sparkContext.master)

In [ ]:
if spark._jsc is None or spark._jsc.sc().isStopped():
    raise RuntimeError(
        "No live Spark session (e.g. after spark.stop()). Restart kernel and run from the top."
    )

df = spark.range(0, 100)
count = df.count()
sample = df.filter(df.id % 17 == 0).limit(5).collect()

assert count == 100, count
print("row count:", count)
print("sample ids (id % 17 == 0):", [r.id for r in sample])

In [ ]:
import os

spark.stop()
os.environ["PYSPARK_EXAMPLE_SESSION_STOPPED"] = "1"
print("OK — Spark stopped. Restart kernel before running Spark cells again.")